In [ ]:
!pip install -q transformers peft accelerate bitsandbytes safetensors torch pandas


In [ ]:
import torch, re, csv, os, math
import xml.etree.ElementTree as ET
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL   = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
ADAPTER_PATH = '/kaggle/input/notebooks/rebeccachenn/model-done/fast/'

print('Files in adapter dir:', os.listdir(ADAPTER_PATH))

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(model, ADAPTER_PATH, is_trainable=False)
model.eval()
model.config.use_cache = True
if hasattr(model, 'gradient_checkpointing_disable'):
    model.gradient_checkpointing_disable()

print('Loaded! Device:', next(model.parameters()).device)


In [ ]:
ALLOWED_TAGS = {
    'svg','g','path','rect','circle','ellipse','line','polyline','polygon',
    'defs','use','symbol','clipPath','mask','linearGradient','radialGradient',
    'stop','text','tspan','title','desc','style','pattern','marker','filter',
}
DISALLOWED_ATTRS_RE = re.compile(
    r'\s+(?:filling|onclick|onload|onmouseover|xmlns:xlink)=["\'][^"\']*("|\')',
    re.IGNORECASE
)
SVG_REGEX = re.compile(r'(<svg\b[\s\S]*?</svg>)', flags=re.IGNORECASE)

def normalize_svg(svg_text):
    if not svg_text: return ''
    svg_text = DISALLOWED_ATTRS_RE.sub('', svg_text)
    for attr in ('width', 'height', 'viewBox'):
        svg_text = re.sub(rf'(<svg\b[^>]*?)\b{attr}=["\'][^"\']*("|\')', r'\1', svg_text)
    svg_text = re.sub(
        r'(<svg\b)',
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"',
        svg_text, count=1
    )
    svg_text = re.sub(r'(xmlns="[^"]*")(?=.*xmlns="[^"]*")', '', svg_text)
    svg_text = re.sub(r'\s+', ' ', svg_text).strip()
    return svg_text

def has_disallowed_tags(svg_text):
    try:
        root = ET.fromstring(svg_text)
        for elem in root.iter():
            tag = elem.tag
            if isinstance(tag, str):
                local = tag.split('}')[-1] if '}' in tag else tag
                if local not in ALLOWED_TAGS: return True
        return False
    except Exception: return True

def is_valid_svg(svg_text):
    if not svg_text: return False
    try:
        root = ET.fromstring(svg_text)
        tag = root.tag.split('}')[-1] if '}' in root.tag else root.tag
        return tag == 'svg'
    except ET.ParseError: return False

def extract_svg(text):
    if not text: return ''
    for tok in ['<|im_end|>', '<|endoftext|>']:
        idx = text.find(tok)
        if idx != -1: text = text[:idx]
    match = SVG_REGEX.search(text)
    if match: return match.group(1).strip()
    start = text.lower().find('<svg')
    if start != -1:
        partial = text[start:].strip()
        if not partial.lower().endswith('</svg>'): partial += '</svg>'
        return partial
    return ''

def fallback_svg():
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="#cccccc"/>'
        '</svg>'
    )

print('Helpers defined.')


In [ ]:
# ── IMPROVEMENT 1: Richer system prompt ──────────────────────────────
# More specific instructions → model stays on-task and avoids wasted tokens
SYSTEM_PROMPT = (
    'You are an expert SVG code generator. '
    'Given a text description, output ONLY a single valid SVG element. '
    'Requirements: '
    'width="256" height="256" viewBox="0 0 256 256", '
    'use only basic SVG shapes and paths, '
    'match the described colors and layout as closely as possible, '
    'no comments, no markdown, no explanations.'
)

test_df = pd.read_csv(
    '/kaggle/input/datasets/rebeccachenn/dlmidterm/test.csv',
    engine='python', quoting=csv.QUOTE_MINIMAL
)
print('test_df shape:', test_df.shape)
test_df.head(2)


## Inference with Quality Improvements

Changes vs previous version:
1. **Richer system prompt** — more specific instructions reduce off-task output
2. **max_new_tokens 350→500** — complex SVGs have room to finish properly
3. **Best-of-N retry** — SVGs that fail greedy decoding are retried with N=3 sampled candidates; the longest valid one is kept (more shapes = better SSIM)
4. **Fixed autocast warning** — `torch.amp.autocast('cuda')`


In [ ]:
BATCH_SIZE     = 4
MAX_NEW_TOKENS = 500   # ✅ more room for complex SVGs to finish
BEST_OF_N      = 3     # ✅ retry failed SVGs with N sampled candidates
SAVE_DIR       = '/kaggle/working/'
os.makedirs(SAVE_DIR, exist_ok=True)

def make_prompt(prompt_text):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt_text},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def generate_batch(prompts, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=1.0):
    """Batched generation. Returns list of (svg, used_fallback)."""
    chat_texts = [make_prompt(p) for p in prompts]
    inputs = tokenizer(
        chat_texts, return_tensors='pt', padding=True,
        truncation=True, max_length=512,
    ).to(model.device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )
    if do_sample:
        gen_kwargs['temperature'] = temperature
        gen_kwargs['top_p'] = 0.95

    with torch.no_grad():
        with torch.amp.autocast('cuda'):  # ✅ fixed deprecation warning
            output_ids = model.generate(**inputs, **gen_kwargs)

    input_len = inputs['input_ids'].shape[1]
    results = []
    for i in range(len(prompts)):
        decoded = tokenizer.decode(output_ids[i][input_len:], skip_special_tokens=False)
        svg = normalize_svg(extract_svg(decoded))
        bad = not is_valid_svg(svg) or has_disallowed_tags(svg) or len(svg) > 15900
        results.append((svg, bad))
    return results


def best_of_n(prompt, n=BEST_OF_N, temperature=0.8):
    """
    ✅ QUALITY IMPROVEMENT: generate N candidates with sampling,
    return the longest valid SVG (more content = better SSIM score).
    Falls back to greedy result if all candidates are invalid.
    """
    candidates = []
    for _ in range(n):
        try:
            res = generate_batch([prompt], do_sample=True, temperature=temperature)
            svg, bad = res[0]
            if not bad:
                candidates.append(svg)
        except RuntimeError:
            torch.cuda.empty_cache()
    if candidates:
        # pick the longest valid SVG — more paths/shapes = higher visual fidelity
        return max(candidates, key=len), False
    # all candidates invalid → greedy fallback
    res = generate_batch([prompt], do_sample=False)
    svg, bad = res[0]
    if bad:
        svg = fallback_svg()
    return svg, bad


# ── Main inference loop ───────────────────────────────────────────────
all_results   = []
fallback_count = 0
retry_count    = 0
n = len(test_df)

for batch_start in range(0, n, BATCH_SIZE):
    batch_rows = test_df.iloc[batch_start : batch_start + BATCH_SIZE]
    prompts = batch_rows['prompt'].tolist()
    ids     = batch_rows['id'].tolist()

    # Step 1: fast greedy batch pass
    try:
        batch_results = generate_batch(prompts)
    except RuntimeError:
        print(f'  OOM at batch {batch_start}, single fallback')
        torch.cuda.empty_cache()
        batch_results = []
        for p in prompts:
            try:
                batch_results.extend(generate_batch([p]))
            except Exception:
                batch_results.append((fallback_svg(), True))

    # Step 2: best-of-N retry for any failed SVGs
    for j, (svg, bad) in enumerate(batch_results):
        if bad:
            retry_count += 1
            svg, bad = best_of_n(prompts[j])
            batch_results[j] = (svg, bad)

    for row_id, (svg, bad) in zip(ids, batch_results):
        if bad: fallback_count += 1
        all_results.append({'id': row_id, 'svg': svg})

    processed = min(batch_start + BATCH_SIZE, n)
    if processed % 100 == 0 or processed == n:
        print(f'Processed {processed}/{n} | fallback:{fallback_count} | retried:{retry_count}')
        part_num = processed // 100
        pd.DataFrame(all_results).to_csv(
            os.path.join(SAVE_DIR, f'submission_partial_{part_num}.csv'), index=False
        )

submission_df = pd.DataFrame(all_results)
print(f'Done! Total:{len(submission_df)} | Fallback:{fallback_count} ({fallback_count/len(submission_df)*100:.1f}%) | Retried:{retry_count}')


In [ ]:
valid_count = submission_df['svg'].apply(is_valid_svg).sum()
print(f'Valid          : {valid_count}/{len(submission_df)} ({valid_count/len(submission_df)*100:.1f}%)')
print(f'Disallowed tags: {submission_df["svg"].apply(has_disallowed_tags).sum()}')
print(f'Over 16000 chars: {(submission_df["svg"].str.len() > 16000).sum()}')

final_path = os.path.join(SAVE_DIR, 'submission.csv')
submission_df.to_csv(final_path, index=False)
print('Saved:', final_path)
submission_df.head()
